In [5]:
import numpy as np

class Softmax:
    # Xavier initialization
    weight_init_factor = 1.

    def forward(self, x):
        exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
        return exp_x / np.sum(exp_x, axis=1, keepdims=True)

class Sigmoid:
    # Xavier initialization
    weight_init_factor = 1.

    def forward(self, x):
        return 1 / (1 + np.exp(-x))
    
    def derivative(self, x):
        s = self.forward(x)
        return s * (1 - s)

class ReLU:
    # He initialization
    weight_init_factor = np.sqrt(2)

    def forward(self, x):
        return np.maximum(0, x)
    
    def derivative(self, x):
        return (x > 0).astype(float)

class Linear:
    # Xavier initialization
    weight_init_factor = 1.0

    def forward(self, x):
        return x

    def derivative(self, x):
        return np.ones_like(x)

In [6]:
class Layer:
    def __init__(self, num_neurons: int, num_input_connections: int, activation_function, rng, alpha=0.05, beta_one=0.9, beta_two=0.999, epsilon = 1e-8):
        self.num_neurons = num_neurons
        self.activation_function = activation_function
        self.weights =  rng.standard_normal((num_input_connections, num_neurons)) * activation_function.weight_init_factor * np.sqrt(1/num_input_connections)
        self.biases = self.biases = rng.standard_normal(num_neurons) * 0.01
        self.prev_inputs = None
        self.z = None
        self.error_signals = None
        self.alpha = alpha


        # adam optimizer stuff
        self.ma_weight_signal = np.zeros((num_input_connections, num_neurons)) # moving average bias gradient signal for adam optimizer
        self.ma_bias_signal = np.zeros(num_neurons) # moving average weight gradient signal for adam optimizer
        self.update_steps = 0
        self.beta_one = beta_one
        self.ma_weight_signal_variance = np.zeros((num_input_connections, num_neurons))
        self.ma_bias_signal_variance = np.zeros(num_neurons)
        self.beta_two = beta_two
        self.epsilon = epsilon

    def forward(self, x) -> np.array:
        self.z = x @ self.weights + self.biases
        self.prev_inputs = x
        return self.activation_function.forward(self.z)

    def backward(self, error_signals) -> np.array:
        self.update_steps += 1

        # i didnt add a derivative to softmax cus im lazy, i didn't feel like spending the time figuring out the jacobians, just passing the loss for the next step all the way through
        if hasattr(self.activation_function, 'derivative'):
            dLdz = error_signals * self.activation_function.derivative(self.z)
        else:
            dLdz = error_signals
    
        batch_size = error_signals.shape[0]

        bias_gradient_signal = np.sum(dLdz, axis=0) / batch_size
        self.ma_bias_signal = (self.beta_one * self.ma_bias_signal) + (1 - self.beta_one) * bias_gradient_signal # first moment
        self.ma_bias_signal_variance = (self.beta_two * self.ma_bias_signal_variance) + (1 - self.beta_two) * bias_gradient_signal ** 2 # second moment
        mb = self.ma_bias_signal / (1 - self.beta_one ** self.update_steps) # bias correction
        vb = self.ma_bias_signal_variance / (1 - self.beta_two ** self.update_steps) # bias correction
        dB = mb / (np.sqrt(vb) + self.epsilon)

        weight_gradient_signal = (self.prev_inputs.T @ dLdz) / batch_size
        self.ma_weight_signal = ( (self.beta_one * self.ma_weight_signal) + (1 - self.beta_one) * weight_gradient_signal ) # first moment
        self.ma_weight_signal_variance = (self.beta_two * self.ma_weight_signal_variance) + (1 - self.beta_two) * weight_gradient_signal ** 2 # second moment
        mw = self.ma_weight_signal / (1 - self.beta_one ** self.update_steps) # bias correction
        vw = self.ma_weight_signal_variance / (1 - self.beta_two ** self.update_steps) # bias correction
        dW = mw / (np.sqrt(vw) + self.epsilon)

        error_signals = dLdz @ self.weights.T
        self.biases += -self.alpha * dB
        self.weights += -self.alpha * dW

        return error_signals


In [ ]:
rng = np.random.default_rng(42)

buffer_capacity = 100_000
state_size = 784 + 784 + 1   # 784 values for the current observed values of all states, 
                             # 784 values in the mask (which pixels we currently have access to), 
                             # and 1 pixel for the remaining budget of pixel reveals left)
num_pixel_reveals = 60
epsilon = 0.9 # for epsilon greedy search

layers = [
    Layer(num_neurons=256, num_input_connections=state_size, activation_function=ReLU(), rng=rng, alpha=0.001),
    Layer(num_neurons=128, num_input_connections=256, activation_function=ReLU(), rng=rng, alpha=0.001),
    Layer(num_neurons=784, num_input_connections=128, activation_function=Linear(), rng=rng, alpha=0.001)
]

def full_forward(x):
    for layer in layers:
        x = layer.forward(x)
    return x

def full_backward(upstream_error):
    for layer in reversed(layers):
        upstream_error = layer.backward(upstream_error)
    return upstream_error

discount_factor = 1
buffer_s = np.zeros((buffer_capacity, state_size))
buffer_a = np.zeros(buffer_capacity)
buffer_r = np.zeros(buffer_capacity)
buffer_s_prime = np.zeros((buffer_capacity, state_size))
buffer_dones = np.zeros(buffer_capacity)

def reveal_pixel(s, a, num_envs):
    return np.zeros((num_envs, state_size)) # replace with revealing one pixel from MNIST image

def evaluate_state(state):
    return 0 # replace with call to the fine-tuned classification MLP

def add_to_buffer(s, a, r, s_prime, done):
    pass

def generate_trajectory_and_populate_buffer(num_envs=32):
    s = np.zeros((num_envs, state_size))
    s[:, -1] = num_pixel_reveals


    for pixel_reveals_left in reversed(range(0, num_pixel_reveals)):
        q_values = full_forward(s) # shape = (num_envs, num actions)
        greedy_actions = np.argmax(q_values, axis=1)  # shape = (num_envs,)

        random_actions = rng.integers(0, 784, size=num_envs)
        mask = rng.random(size=num_envs) < epsilon
        a = np.where(mask, random_actions, greedy_actions) # apply the mask (epsilon greedy search)


        s_prime = reveal_pixel(s, a, num_envs)
        s_prime[:, -1] = pixel_reveals_left - 1 

        if pixel_reveals_left == 0:
            s_prime[:, -1] = 0
            add_to_buffer(s, a, evaluate_state(s_prime), s_prime, np.ones(num_envs))
        else:
            add_to_buffer(s, a, np.zeros(num_envs), s_prime, np.zeros(num_envs))
        s = s_prime



In [ ]:
batch_size = 128

indices = rng.choice(buffer_capacity, size=batch_size, replace=False)
batch_s = buffer_s[indices]
batch_a = buffer_a[indices]
batch_r = buffer_r[indices]
batch_s_prime = buffer_s_prime[indices]
batch_dones = buffer_dones[indices]

q_values_s = full_forward(batch_s)
q_sa = q_values_s[np.arange(batch_size), batch_a.astype(int)] # what was the q value for the action we chose
loss_grad =  q_sa - ( batch_r + (1-batch_dones) * discount_factor * np.max(full_forward(batch_s_prime), axis=1) )

# the mlp needs gradients in the shape (batch size, amount of mlp outputs)
filled_grads = np.zeros((batch_size, 784))
filled_grads[np.arange(batch_size), batch_a.astype(int)] = loss_grad

full_backward(filled_grads)

array([[-8.67859175e-04, -6.79467155e-04,  3.26888982e-05, ...,
         1.97619757e-04, -6.27941324e-04,  5.20631330e-04],
       [-8.67859175e-04, -6.79467155e-04,  3.26888982e-05, ...,
         1.97619757e-04, -6.27941324e-04,  5.20631330e-04],
       [-8.67859175e-04, -6.79467155e-04,  3.26888982e-05, ...,
         1.97619757e-04, -6.27941324e-04,  5.20631330e-04],
       ...,
       [-8.67859175e-04, -6.79467155e-04,  3.26888982e-05, ...,
         1.97619757e-04, -6.27941324e-04,  5.20631330e-04],
       [-8.67859175e-04, -6.79467155e-04,  3.26888982e-05, ...,
         1.97619757e-04, -6.27941324e-04,  5.20631330e-04],
       [-8.67859175e-04, -6.79467155e-04,  3.26888982e-05, ...,
         1.97619757e-04, -6.27941324e-04,  5.20631330e-04]],
      shape=(128, 1569))